'''
Graded Challenge 6

Nama  : Zafirah Aida Adista

Batch : CODA-RMT-015

Program ini dibuat untuk melakukan evaluasi konsep ETL dan Data Modeling 

'''

In [2]:
# Mengimpor SparkSession
from pyspark.sql import SparkSession
# Mengimpor berbagai fungsi PySpark untuk pemrosesan data
from pyspark.sql.functions import (
    col,lower, regexp_replace, sum, count,
    to_date, date_format, dayofmonth, month, quarter, year, dayofweek,
    monotonically_increasing_id)

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("WriteToPostgres") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()

# Load Dataset

In [3]:
# Membaca dataset
users = spark.read.csv("Dataset GC 6/users.csv", header=True, inferSchema=True)
products = spark.read.csv("Dataset GC 6/products.csv", header=True, encoding="UTF-8", inferSchema=True)
orders = spark.read.csv("Dataset GC 6/orders.csv", header=True, inferSchema=True)
order_items = spark.read.csv("Dataset GC 6/order_items.csv", header=True, inferSchema=True)
distribution_centers = spark.read.csv("Dataset GC 6/distribution_centers.csv", header=True, inferSchema=True)
inventory = spark.read.csv("Dataset GC 6/inventory.csv", header=True, inferSchema=True)

In [4]:
# Melihat preview dataset
users.show(5)
products.show(5)
orders.show(5)
order_items.show(5)
distribution_centers.show(5)
inventory.show(5)

+-----+----------+---------+--------------------+---+------+-----+--------------------+-----------+----+-------+------------+------------+--------------+-------------------+--------------------+
|   id|first_name|last_name|               email|age|gender|state|      street_address|postal_code|city|country|    latitude|   longitude|traffic_source|         created_at|           user_geom|
+-----+----------+---------+--------------------+---+------+-----+--------------------+-----------+----+-------+------------+------------+--------------+-------------------+--------------------+
|57902|   Monique|   Jacobs|moniquejacobs@exa...| 23|     F| Acre|5819 Weber Court ...|  69980-000|null| Brasil|-8.065346116|-72.87094866|        Search|2019-09-09 15:40:00|POINT(-72.8709486...|
|61864|      Erik|    Berry|erikberry@example...| 24|     M| Acre|59978 Willis Lodg...|  69980-000|null| Brasil|-8.065346116|-72.87094866|        Search|2019-07-03 01:16:00|POINT(-72.8709486...|
|66378|     Shawn|      A

In [68]:
# Simple Data Exploration

In [5]:
# Melihat struktur data

users.printSchema()
products.printSchema()
orders.printSchema()
order_items.printSchema()
distribution_centers.printSchema()
inventory.printSchema()

root
 |-- id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- state: string (nullable = true)
 |-- street_address: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- traffic_source: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- user_geom: string (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- cost: double (nullable = true)
 |-- category: string (nullable = true)
 |-- name: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- retail_price: double (nullable = true)
 |-- department: string (nullable = true)
 |-- sku: string (nullable = true)
 |-- distribution_center_id: integer (nul

In [6]:
# Melihat summary deskripsi statistik

users.describe().show()
products.describe().show()
orders.describe().show()
order_items.describe().show()
distribution_centers.describe().show()
inventory.describe().show()

+-------+-----------------+----------+---------+--------------------+------------------+------+-------------+-------------------+------------------+------+-------------+------------------+------------------+--------------+--------------------+
|summary|               id|first_name|last_name|               email|               age|gender|        state|     street_address|       postal_code|  city|      country|          latitude|         longitude|traffic_source|           user_geom|
+-------+-----------------+----------+---------+--------------------+------------------+------+-------------+-------------------+------------------+------+-------------+------------------+------------------+--------------+--------------------+
|  count|           100000|    100000|   100000|              100000|            100000|100000|       100000|             100000|            100000|100000|       100000|            100000|            100000|        100000|              100000|
|   mean|          50000

In [8]:
# Melihat jumlah rows pada dataset

print(f"Users: {users.count()}")
print(f"Products: {products.count()}")
print(f"Orders: {orders.count()}")
print(f"Order_items: {order_items.count()}")
print(f"Distribution_centers: {distribution_centers.count()}")
print(f"Inventory: {inventory.count()}")

Users: 100000
Products: 29120
Orders: 124918
Order_items: 181110
Distribution_centers: 10
Inventory: 29049


# Data Cleaning

## Menghapus kolom yang tidak dibutuhkan

Kolom yang tidak diperlukan pada masing-masing tabel karena redundant dan/ tidak berhubungan dengan sales:
- Users: street_address, postal_code, latitude, longitude, user_geom
- Products: sku
- Orders: gender,status, returned_at, shipped_at, delivered_at, num_of_item
- Order_items: inventory_item_id, shipped_at, delivered_at, returned_at
- Distribution_centers: distribution_center_geom

In [9]:
# Menghapus kolom pada tabel users
users_clean = users.drop(
    "street_address",
    "postal_code",
    "latitude",
    "longitude",
    "user_geom")

# Menghapus kolom pada tabel products
products_clean = products.drop("sku")

# Menghapus kolom pada tabel orders
orders_clean = orders.drop(
    "gender",
    "status",
    "returned_at",
    "shipped_at",
    "delivered_at",
    "num_of_item")

# Menghapus kolom pada tabel order_items
order_items_clean = order_items.drop(
    "inventory_item_id",
    "shipped_at",
    "delivered_at",
    "returned_at")

# Menghapus kolom pada tabel distribution_centers
distribution_centers_clean = distribution_centers.drop("distribution_center_geom")

## Missing values

In [10]:
# Melihat nilai missing value

users_clean.select([sum(col(c).isNull().cast("int")).alias(c) for c in users_clean.columns]).show()
products_clean.select([sum(col(c).isNull().cast("int")).alias(c) for c in products_clean.columns]).show()
orders_clean.select([sum(col(c).isNull().cast("int")).alias(c) for c in orders_clean.columns]).show()
order_items_clean.select([sum(col(c).isNull().cast("int")).alias(c) for c in order_items_clean.columns]).show()
distribution_centers_clean.select([sum(col(c).isNull().cast("int")).alias(c) for c in distribution_centers_clean.columns]).show()
inventory.select([sum(col(c).isNull().cast("int")).alias(c) for c in inventory.columns]).show()

+---+----------+---------+-----+---+------+-----+----+-------+--------------+----------+
| id|first_name|last_name|email|age|gender|state|city|country|traffic_source|created_at|
+---+----------+---------+-----+---+------+-----+----+-------+--------------+----------+
|  0|         0|        0|    0|  0|     0|    0|   0|      0|             0|         0|
+---+----------+---------+-----+---+------+-----+----+-------+--------------+----------+

+---+----+--------+----+-----+------------+----------+----------------------+
| id|cost|category|name|brand|retail_price|department|distribution_center_id|
+---+----+--------+----+-----+------------+----------+----------------------+
|  0|   0|       0|   2|   24|           0|         0|                     0|
+---+----+--------+----+-----+------------+----------+----------------------+

+--------+-------+----------+
|order_id|user_id|created_at|
+--------+-------+----------+
|       0|      0|         0|
+--------+-------+----------+

+---+-------

In [11]:
# Handling missing values pada kolom name

# karena hanya terdapat 2 rows yang missing maka handlingnya adalah drop rows tersebut
products_clean = products_clean.dropna(subset=["name"])

In [12]:
# Handling missing values pada kolom brand

# Mengisi dengan nilai "unknown" pada rows yang missing
products_clean = products_clean.fillna({"brand":"Unknown"})

In [13]:
# Mengecek dataset setelah handling missing values
products_clean.select([sum(col(c).isNull().cast("int")).alias(c) for c in products_clean.columns]).show()

+---+----+--------+----+-----+------------+----------+----------------------+
| id|cost|category|name|brand|retail_price|department|distribution_center_id|
+---+----+--------+----+-----+------------+----------+----------------------+
|  0|   0|       0|   0|    0|           0|         0|                     0|
+---+----+--------+----+-----+------------+----------+----------------------+



In [ ]:
Missing values pada tabel products sudah teratasi

## "Null" Values

In [14]:
# Mengecek rows yang mengandung variasi string "null"

# Dataframe yang ingin dicek
datasets = {
    "Users": users_clean,
    "Products": products_clean,
    "Orders": orders_clean,
    "Order Items": order_items_clean,
    "Distribution Centers": distribution_centers_clean,
    "Inventory": inventory}

# iterasi setiap dataset untuk mengidentifikasi rows yg mengandung variasi string "null"
for name, df in datasets.items():
    print(f"--- Checking 'nul-like' in: {name} ---")
    
    # Mencari teks yang mengandung "null" (seperti nules, null, nul)
    df.select([sum((lower(col(c)).like("%nul%")).cast("int")).alias(c) 
        for c in df.columns]).show()


--- Checking 'nul-like' in: Users ---
+---+----------+---------+-----+---+------+-----+----+-------+--------------+----------+
| id|first_name|last_name|email|age|gender|state|city|country|traffic_source|created_at|
+---+----------+---------+-----+---+------+-----+----+-------+--------------+----------+
|  0|         0|        0|    0|  0|     0|    0| 992|      0|             0|         0|
+---+----------+---------+-----+---+------+-----+----+-------+--------------+----------+

--- Checking 'nul-like' in: Products ---
+---+----+--------+----+-----+------------+----------+----------------------+
| id|cost|category|name|brand|retail_price|department|distribution_center_id|
+---+----+--------+----+-----+------------+----------+----------------------+
|  0|   0|       0|   0|    0|           0|         0|                     0|
+---+----+--------+----+-----+------------+----------+----------------------+

--- Checking 'nul-like' in: Orders ---
+--------+-------+----------+
|order_id|user_

In [15]:
# Melihat variasi teks "null" di kolom city
users_clean.select("city").distinct().filter(col("city").rlike("(?i).*nul.*")).show()

+-----+
| city|
+-----+
|Nules|
| null|
+-----+



In [16]:
# Replace variasi teks "null" menjadi "unknown"
users_clean = users_clean.withColumn("city", regexp_replace(col("city"), "nules", "Unknown"))
users_clean = users_clean.withColumn("city", regexp_replace(col("city"), "nul", "Unknown"))

users_clean.show(5)

+-----+----------+---------+--------------------+---+------+-----+--------+-------+--------------+-------------------+
|   id|first_name|last_name|               email|age|gender|state|    city|country|traffic_source|         created_at|
+-----+----------+---------+--------------------+---+------+-----+--------+-------+--------------+-------------------+
|57902|   Monique|   Jacobs|moniquejacobs@exa...| 23|     F| Acre|Unknownl| Brasil|        Search|2019-09-09 15:40:00|
|61864|      Erik|    Berry|erikberry@example...| 24|     M| Acre|Unknownl| Brasil|        Search|2019-07-03 01:16:00|
|66378|     Shawn|      Ali|shawnali@example.org| 24|     M| Acre|Unknownl| Brasil|        Search|2022-11-17 01:42:00|
|87473|       Amy|    White|amywhite@example.org| 16|     F| Acre|Unknownl| Brasil|        Search|2019-11-27 12:22:00|
|69167|   Michael|   Macias|michaelmacias@exa...| 50|     M| Acre|Unknownl| Brasil|        Search|2026-01-27 11:50:00|
+-----+----------+---------+--------------------

## Duplicate Values

In [17]:
# Mengecek duplicates value

for name, df in datasets.items():
    total = df.count()
    unique = df.dropDuplicates().count()
    duplicate = total - unique
    print(f"{name}: Total={total} | Unique={unique} | Duplicates={duplicate}")


Users: Total=100000 | Unique=100000 | Duplicates=0
Products: Total=29118 | Unique=29118 | Duplicates=0
Orders: Total=124918 | Unique=124918 | Duplicates=0
Order Items: Total=181110 | Unique=181110 | Duplicates=0
Distribution Centers: Total=10 | Unique=10 | Duplicates=0
Inventory: Total=29049 | Unique=29049 | Duplicates=0


Tidak ada dupicates values pada masing-masing kolom

# Transform

In [18]:
# Pembuatan tabel Dim_Customers

dim_customer = (users_clean # Diambil dari dataset users
    .select("id", "first_name", "last_name", "email", "age", "gender", "traffic_source") # memilih kolom yang digunakan pada tabel dim_customer
    .withColumnRenamed("id", "customer_id")) # rename kolom

# Melihat preview tabel
dim_customer.show(5) 

+-----------+----------+---------+--------------------+---+------+--------------+
|customer_id|first_name|last_name|               email|age|gender|traffic_source|
+-----------+----------+---------+--------------------+---+------+--------------+
|      57902|   Monique|   Jacobs|moniquejacobs@exa...| 23|     F|        Search|
|      61864|      Erik|    Berry|erikberry@example...| 24|     M|        Search|
|      66378|     Shawn|      Ali|shawnali@example.org| 24|     M|        Search|
|      87473|       Amy|    White|amywhite@example.org| 16|     F|        Search|
|      69167|   Michael|   Macias|michaelmacias@exa...| 50|     M|        Search|
+-----------+----------+---------+--------------------+---+------+--------------+
only showing top 5 rows


In [19]:
# Pembuatan tabel Dim_geography

dim_geography = (users_clean. # diambil dari dataset users
    select("city", "state", "country") # memilih kolom yang digunakan pada tabel dim_geography
    .withColumn("geography_id", monotonically_increasing_id())) # 

# Melihat preview tabel
dim_geography.show(5)

+--------+-----+-------+------------+
|    city|state|country|geography_id|
+--------+-----+-------+------------+
|Unknownl| Acre| Brasil|           0|
|Unknownl| Acre| Brasil|           1|
|Unknownl| Acre| Brasil|           2|
|Unknownl| Acre| Brasil|           3|
|Unknownl| Acre| Brasil|           4|
+--------+-----+-------+------------+
only showing top 5 rows


In [20]:
# Pembuatan tabel Dim_Product

dim_product = (products_clean # Diambil dari dataset products
    .select("id", "name", "category", "brand", "department", "cost", "retail_price") # memilih kolom yang digunakan pada tabel dim_product
    .withColumnRenamed("id", "product_id"))  # rename kolom

# Melihat preview tabel
dim_product.show(5)

+----------+--------------------+-----------+-----+----------+------------------+------------------+
|product_id|                name|   category|brand|department|              cost|      retail_price|
+----------+--------------------+-----------+-----+----------+------------------+------------------+
|     13842|Low Profile Dyed ...|Accessories|   MG|     Women| 2.518749990849756|              6.25|
|     13928|Low Profile Dyed ...|Accessories|   MG|     Women|2.3383499148894105| 5.949999809265137|
|     14115|Enzyme Regular So...|Accessories|   MG|     Women| 4.879559879379869|10.989999771118164|
|     14157|Enzyme Regular So...|Accessories|   MG|     Women| 4.648769887297898|10.989999771118164|
|     14273|Washed Canvas Ivy...|Accessories|   MG|     Women| 6.507929886473045|15.989999771118164|
+----------+--------------------+-----------+-----+----------+------------------+------------------+
only showing top 5 rows


In [21]:
# Pembuatan tabel Dim_distribution_center

dim_distribution_center = (distribution_centers_clean # diambil dari dataset distribution_centers
    .select("id", "name", "latitude", "longitude") # memilih kolom yang digunakan pada tabel dim_distribution_center
    .withColumnRenamed("id", "distribution_center_id")) # rename kolom

# Melihat preview tabel
dim_distribution_center.show(5)

+----------------------+-------------+--------+---------+
|distribution_center_id|         name|latitude|longitude|
+----------------------+-------------+--------+---------+
|                     8|    Mobile AL| 30.6944| -88.0431|
|                     9|Charleston SC| 32.7833| -79.9333|
|                     1|   Memphis TN| 35.1174| -89.9711|
|                    10|  Savannah GA| 32.0167| -81.1167|
|                     3|   Houston TX| 29.7604| -95.3698|
+----------------------+-------------+--------+---------+
only showing top 5 rows


In [22]:
# Pembuatan tabel Dim_date
dim_date = (orders_clean # diambil dari dataset orders
    .select(to_date("created_at").alias("order_date")) # Mengubah format menjadi tipe data Date
    .dropDuplicates(["order_date"]) # 
    .withColumn("date_id",     date_format("order_date", "yyyyMMdd").cast("int")) # Membuat PK "date_id" dalam format integer YYYYMMDD
    .withColumn("day",         dayofmonth("order_date")) # Ekstrak hari dari kolom "order_date"
    .withColumn("month_num",   month("order_date")) # Ekstrak nomor bulan dari kolom "order_date"
    .withColumn("month_name",  date_format("order_date", "MMMM")) # Ekstrak nama bulan dari kolom "order_date"
    .withColumn("quarter",     quarter("order_date")) # Ekstrak kuartal tahunan dari kolom "order_date"
    .withColumn("year",        year("order_date")) # Ekstrak tahun dari kolom "order_date"
    .withColumn("day_of_week", date_format("order_date", "EEEE")) # Ekstrak nama hari dari kolom "order_date"
    .withColumn("is_weekend",  dayofweek("order_date").isin(1, 7))) #Identifikasi weekend

# Melihat preview tabel
dim_date.show(5)

+----------+--------+---+---------+----------+-------+----+-----------+----------+
|order_date| date_id|day|month_num|month_name|quarter|year|day_of_week|is_weekend|
+----------+--------+---+---------+----------+-------+----+-----------+----------+
|2022-07-31|20220731| 31|        7|      July|      3|2022|     Sunday|      true|
|2025-10-21|20251021| 21|       10|   October|      4|2025|    Tuesday|     false|
|2024-09-18|20240918| 18|        9| September|      3|2024|  Wednesday|     false|
|2026-02-13|20260213| 13|        2|  February|      1|2026|     Friday|     false|
|2025-02-16|20250216| 16|        2|  February|      1|2025|     Sunday|      true|
+----------+--------+---+---------+----------+-------+----+-----------+----------+
only showing top 5 rows


In [23]:
# Pembuatan tabel fact_sales
fact_sales = (
    order_items_clean.alias("oi") # Dataset sumber utama
    .join(orders_clean.alias("o"),   "order_id") # Join dengan tabel orders berdasarkan order_id
    .join(products_clean.alias("p"), order_items_clean["product_id"] == products_clean["id"]) # Join dengan tabel products berdasarkan ID produk 
    .join(users_clean.alias("u"),    orders_clean["user_id"] == users_clean["id"]) # Join dengan tabel users berdasarkan ID user
    .join(inventory.alias("inv"),    "product_id") # Join dengan inventory berdasarkan product_id
    .join(dim_geography.alias("g"),     (col("u.city") == col("g.city")) & (col("u.country") == col("g.country")))
    .groupBy( # grouping berdasarkan key dimmension dan key atribute
        col("o.order_id"),                                                             # FK
        col("o.user_id").alias("customer_id"),                                         # FK dim_customer
        col("p.id").alias("product_id"),                                               # FK dim_product
        col("p.cost"),                                                                 # dipakai untuk hitung cost_amount lalu di-drop
        date_format(to_date("o.created_at"), "yyyyMMdd").cast("int").alias("date_id"), # Membuat date_id dalam format integer YYYYMMDD 
        col("g.geography_id"),                                                                 # FK dim_geography
        col("inv.product_distribution_center_id").alias("distribution_center_id"),     # FK dim_distribution_centers
        col("oi.status"),)
    .agg(count("oi.id").alias("quantity"),            # Menghitung quantity berdasarkan kombinasi order_id + product_id yang sama
         sum("oi.sale_price").alias("sales_amount"),) # Menghitung total harga jual berdasarkan kombinasi order_id + product_id yang sama
    .withColumn("cost_amount", col("quantity") * col("cost")) # Menghitung total cost dengan qty * cost per unit dari dim_product
    .drop("cost") # sudah tidak dibutuhkan setelah cost_amount dihitung
    .withColumn("sales_id", monotonically_increasing_id())) # Membuat PK tabel fact_sales untuk setiap baris

# Melihat preview tabel
fact_sales.show(5)

+--------+-----------+----------+--------+------------+----------------------+----------+--------+------------------+-----------------+--------+
|order_id|customer_id|product_id| date_id|geography_id|distribution_center_id|    status|quantity|      sales_amount|      cost_amount|sales_id|
+--------+-----------+----------+--------+------------+----------------------+----------+--------+------------------+-----------------+--------+
|       1|          1|     18949|20240812|        9820|                     6|Processing|       1|  59.9900016784668|30.17497084583316|       0|
|       1|          1|     18949|20240812|        9827|                     6|Processing|       1|  59.9900016784668|30.17497084583316|       1|
|       1|          1|     23510|20240812|        9809|                     2|Processing|       1|58.950000762939446| 30.8898004120782|       2|
|       2|          1|     18167|20260109|        9806|                     1|   Shipped|       1|              25.0| 9.3500000424

In [28]:
# Cek tipe data sebelum load ke postgres
dim_customer.printSchema()
dim_geography.printSchema()
dim_product.printSchema()
dim_distribution_center.printSchema()
dim_date.printSchema()
fact_sales.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- traffic_source: string (nullable = true)

root
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- geography_id: integer (nullable = false)

root
 |-- product_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = false)
 |-- department: string (nullable = true)
 |-- cost: double (nullable = true)
 |-- retail_price: double (nullable = true)

root
 |-- distribution_center_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)

root
 |-- order_date: date (nullable = true)
 |-- date_id: integer (nullable = true

In [27]:
# Mengubah tipe data pada kolom "geography_id" di tabel dim_geography
dim_geography = (dim_geography.withColumn("geography_id", col("geography_id").cast("int")))

# Mengubah tipe data pada kolom "geography_id" dan "quantity" di tabel fact_sales 
fact_sales = (fact_sales
    .withColumn("geography_id", col("geography_id").cast("int"))
    .withColumn("quantity", col("quantity").cast("int")))

# Load Data ke Postgres

In [24]:
# PostgreSQL JDBC Connection
postgres_url = "jdbc:postgresql://host.docker.internal:5432/GC6"
postgres_properties = {
    "user": "postgres",
    "password": "12345",
    "driver": "org.postgresql.Driver"
}

# Write DataFrame to PostgreSQL
dim_customer.write.jdbc(url=postgres_url, table="customer", mode="overwrite", properties=postgres_properties)
dim_geography.write.jdbc(url=postgres_url, table="geography", mode="overwrite", properties=postgres_properties)
dim_product.write.jdbc(url=postgres_url, table="products", mode="overwrite", properties=postgres_properties)
dim_distribution_center.write.jdbc(url=postgres_url, table="distribution_centers", mode="overwrite", properties=postgres_properties)
dim_date.write.jdbc(url=postgres_url, table="date", mode="overwrite", properties=postgres_properties)
fact_sales.write.jdbc(url=postgres_url, table="sales", mode="overwrite", properties=postgres_properties)


# Link Google Slides:
https://docs.google.com/presentation/d/1AtPomkvH7Xk3ZEZ0K6BVijzSTxC4T-Eg0HNWtO-Sp0g/edit?usp=sharing